# Variance Correction via Laplace Approximation

Loads parquet outputs from CS1 (linear), CS4 (hierarchical linear), CS5 (hierarchical logistic).  
Accepts the CAVI posterior mean as given, then recovers the correct posterior covariance via  
the Laplace approximation: $\boldsymbol{\Sigma}_{\text{corr}} = (-H)^{-1}$ where $H$ is the  
Hessian of $\log p(\boldsymbol{\theta}|\mathbf{y})$ evaluated at the CAVI mean.

Resulting CIs are compared against the Gibbs gold standard.

**Stage 0** — Setup  
**Stage 1** — CS1: Bayesian linear regression  
**Stage 2** — CS4: Hierarchical Bayesian linear regression  
**Stage 3** — CS5: Hierarchical Bayesian logistic regression  
**Stage 4** — Cross-case summary

In [1]:
# Stage 0 — Setup
import sys
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from scipy.special import digamma, gammaln, expit
import time

ROOT   = Path('..').resolve()
FIGS   = ROOT / 'figs'
DATA   = ROOT / 'results' / 'data'
TABLES = ROOT / 'results' / 'tables'
for d in [FIGS, DATA, TABLES]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 82171165
np.random.seed(SEED)

sys.path.insert(0, str(ROOT / 'python'))
from vb_utils_py import set_pub_style, pub_colours
set_pub_style()
COLOURS = pub_colours()

def save_fig(fig, name):
    path = FIGS / name
    fig.savefig(path, dpi=300, bbox_inches='tight', pad_inches=0.05)
    print(f'  Saved: {path.name}')
    plt.close(fig)
    return path

def hessian_fd(fn, theta, h=1e-4):
    """Central-difference Hessian of scalar fn at theta."""
    D = len(theta)
    H = np.zeros((D, D))
    f0 = fn(theta)
    for i in range(D):
        ei = np.zeros(D); ei[i] = h
        H[i, i] = (fn(theta + ei) - 2*f0 + fn(theta - ei)) / h**2
        for j in range(i+1, D):
            ej = np.zeros(D); ej[j] = h
            H[i, j] = H[j, i] = (fn(theta + ei + ej) - fn(theta + ei - ej)
                                  - fn(theta - ei + ej) + fn(theta - ei - ej)) / (4*h**2)
    return H

def laplace_sigma(fn, theta, h=1e-4):
    """Sigma_corr = (-H)^{-1} via Laplace approximation."""
    H = hessian_fd(fn, theta, h=h)
    return np.linalg.inv(-H), H

print('Setup complete. ROOT =', ROOT)

Publication style applied  (IEEE-compatible, serif fonts, 300 dpi).
Setup complete. ROOT = D:\github\ENEL445


---
## Stage 1 — CS1: Bayesian Linear Regression

Model: $y_i = \mathbf{x}_i^T\boldsymbol{\beta} + \varepsilon_i$,  
$\varepsilon_i \sim \mathcal{N}(0, \tau_e^{-1})$, flat priors on $\boldsymbol{\beta}$ and $\tau_e$.

Log-posterior in unconstrained space $(\boldsymbol{\beta},\,\ell_e = \log\tau_e)$:
$$\log p(\boldsymbol{\beta}, \ell_e \mid \mathbf{y}) = \left(\tfrac{N}{2}+1\right)\ell_e - \tfrac{e^{\ell_e}}{2}\|\mathbf{y} - \mathbf{X}\boldsymbol{\beta}\|^2$$

In [2]:
# Stage 1a — Load CS1 data and run CAVI
df1 = pl.read_parquet(DATA / 'linear_data.parquet')
X1  = np.column_stack([df1['X0'].to_numpy(), df1['X1'].to_numpy()])
y1  = df1['y'].to_numpy()
N1, p1 = X1.shape

from vb_algorithms_py import SimpleLinearVB
vb1       = SimpleLinearVB(X1, y1)         # flat priors
res1      = vb1.fit(max_iter=500, tol=1e-10)
mu1       = res1['mu_beta']                # CAVI posterior mean for beta
Sigma1_vb = res1['Sigma_beta']             # CAVI variational covariance
E_tau1    = res1['E_tau_e']
a1, b1    = res1['a_e_new'], res1['b_e_new']

# Load Gibbs reference
df1g = pl.read_parquet(DATA / 'linear_gibbs.parquet')
gibbs1_beta0 = df1g['beta0'].to_numpy()
gibbs1_beta1 = df1g['beta1'].to_numpy()
gibbs1_tau_e = df1g['tau_e'].to_numpy()
g1_mu  = {'beta0': gibbs1_beta0.mean(), 'beta1': gibbs1_beta1.mean(), 'tau_e': gibbs1_tau_e.mean()}
g1_sd  = {'beta0': gibbs1_beta0.std(ddof=1), 'beta1': gibbs1_beta1.std(ddof=1), 'tau_e': gibbs1_tau_e.std(ddof=1)}

print(f'CS1 CAVI:  mu_beta={mu1.round(4)},  E[tau_e]={E_tau1:.4f}')
print(f'CS1 CAVI:  SD_beta0={np.sqrt(Sigma1_vb[0,0]):.4f},  SD_beta1={np.sqrt(Sigma1_vb[1,1]):.4f}')
print(f'CS1 Gibbs: mu_beta0={g1_mu["beta0"]:.4f},  mu_beta1={g1_mu["beta1"]:.4f}')
print(f'CS1 Gibbs: SD_beta0={g1_sd["beta0"]:.4f},  SD_beta1={g1_sd["beta1"]:.4f}')
print(f'CS1 SD ratios (CAVI/Gibbs): beta0={np.sqrt(Sigma1_vb[0,0])/g1_sd["beta0"]:.3f},  beta1={np.sqrt(Sigma1_vb[1,1])/g1_sd["beta1"]:.3f}')

VB converged after 10 iterations
CS1 CAVI:  mu_beta=[2.0614 1.4987],  E[tau_e]=6.4074
CS1 CAVI:  SD_beta0=0.1101,  SD_beta1=0.0379
CS1 Gibbs: mu_beta0=2.0616,  mu_beta1=1.4984
CS1 Gibbs: SD_beta0=0.1124,  SD_beta1=0.0387
CS1 SD ratios (CAVI/Gibbs): beta0=0.979,  beta1=0.981


In [3]:
# Stage 1b — Laplace correction for CS1

def log_post_cs1(theta):
    """Log-posterior for CS1 in unconstrained space (beta0, beta1, log_tau_e).
    Flat priors on beta and tau_e.
    """
    beta  = theta[:2]
    ell_e = theta[2]
    tau_e = np.exp(ell_e)
    r = y1 - X1 @ beta
    # (N/2 + 1)*ell_e  - tau_e/2 * ||r||^2
    # The +1 comes from the Jacobian (flat prior on tau_e => flat-in-log has Jacobian tau_e)
    return (N1/2 + 1) * ell_e - tau_e/2 * np.dot(r, r)

# CAVI expansion point in unconstrained space
theta1_cavi = np.array([mu1[0], mu1[1], np.log(E_tau1)])

t0 = time.perf_counter()
Sigma1_corr, H1 = laplace_sigma(log_post_cs1, theta1_cavi, h=1e-4)
t1 = time.perf_counter()

sd1_vb    = np.sqrt(np.diag(Sigma1_vb))        # CAVI SD for beta
sd1_corr  = np.sqrt(np.diag(Sigma1_corr)[:2])  # Laplace SD for beta (first 2 params)
sd1_gibbs = np.array([g1_sd['beta0'], g1_sd['beta1']])

print(f'CS1 Laplace correction ({t1-t0:.2f} s):')
print(f'  SD_beta0: CAVI={sd1_vb[0]:.4f},  Laplace={sd1_corr[0]:.4f},  Gibbs={sd1_gibbs[0]:.4f}')
print(f'  SD_beta1: CAVI={sd1_vb[1]:.4f},  Laplace={sd1_corr[1]:.4f},  Gibbs={sd1_gibbs[1]:.4f}')
print(f'  SD ratio (Laplace/Gibbs): beta0={sd1_corr[0]/sd1_gibbs[0]:.3f},  beta1={sd1_corr[1]/sd1_gibbs[1]:.3f}')
print(f'  SD ratio (CAVI/Gibbs):    beta0={sd1_vb[0]/sd1_gibbs[0]:.3f},  beta1={sd1_vb[1]/sd1_gibbs[1]:.3f}')

CS1 Laplace correction (0.00 s):
  SD_beta0: CAVI=0.1101,  Laplace=0.1101,  Gibbs=0.1124
  SD_beta1: CAVI=0.0379,  Laplace=0.0379,  Gibbs=0.0387
  SD ratio (Laplace/Gibbs): beta0=0.979,  beta1=0.981
  SD ratio (CAVI/Gibbs):    beta0=0.979,  beta1=0.981


In [4]:
# Stage 1c — CS1 CI comparison figure
param_names1 = [r'$\beta_0$', r'$\beta_1$']
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for i, pname in enumerate(param_names1):
    ax = axes[i]
    gname = 'beta0' if i == 0 else 'beta1'
    g_samp = gibbs1_beta0 if i == 0 else gibbs1_beta1
    
    x_lo = g_samp.mean() - 4*g1_sd[gname]
    x_hi = g_samp.mean() + 4*g1_sd[gname]
    xv = np.linspace(x_lo, x_hi, 400)

    ax.hist(g_samp, bins=60, density=True, color='#E7298A', alpha=0.30,
            label='Gibbs', edgecolor='none')
    ax.plot(xv, stats.norm.pdf(xv, mu1[i], sd1_vb[i]),
            lw=1.8, color='steelblue', ls='--', label='CAVI (collapsed)')
    ax.plot(xv, stats.norm.pdf(xv, mu1[i], sd1_corr[i]),
            lw=2.0, color='darkorange', ls='-', label='Laplace corrected')

    ax.set_xlabel(pname, fontsize=12)
    ax.set_ylabel('Density')
    ax.set_title(f'CS1 — posterior {pname}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('CS1 (linear regression): CAVI vs Laplace vs Gibbs', fontsize=11, y=1.01)
plt.tight_layout()
save_fig(fig, 'correction_cs1_posterior.png')
print('Saved: correction_cs1_posterior.png')

  Saved: correction_cs1_posterior.png
Saved: correction_cs1_posterior.png


---
## Stage 2 — CS4: Hierarchical Bayesian Linear Regression

Model: $y_{ij} = \mathbf{x}_{ij}^T\boldsymbol{\beta} + u_j + \varepsilon_{ij}$,  
$u_j \sim \mathcal{N}(0, \tau_u^{-1})$, $\boldsymbol{\beta} \sim \mathcal{N}(0, \lambda^{-1}I)$,  
$\tau_e \sim \text{Gamma}(\alpha_e, \gamma_e)$, $\tau_u \sim \text{Gamma}(\alpha_u, \gamma_u)$.  
Parameters: $\lambda=0.1$, $\alpha_e=\alpha_u=1$, $\gamma_e=\gamma_u=0.1$.

Unconstrained: $(\boldsymbol{\beta}, \mathbf{u}, \ell_e, \ell_u)$ — 9D.

In [5]:
# Stage 2a — Load CS4 data and run CAVI
df4 = pl.read_parquet(DATA / 'hierarchical_linear_data.parquet')
X4      = np.column_stack([df4['X0'].to_numpy(), df4['X1'].to_numpy()])
y4      = df4['y'].to_numpy()
groups4 = df4['group'].to_numpy()
N4, p4  = X4.shape
J4      = int(groups4.max()) + 1
n_groups4 = np.array([np.sum(groups4 == j) for j in range(J4)])

# Prior hyperparameters (same as CS4 notebook)
LAM4    = 0.1
ALPHA_E4, GAMMA_E4 = 1.0, 0.1
ALPHA_U4, GAMMA_U4 = 1.0, 0.1

def _warm_cavi_cs4(X, y, groups, J, n_groups, n_iter=120):
    N, pp = X.shape
    mu_beta = np.linalg.lstsq(X, y, rcond=None)[0]
    Sigma_beta = 0.01 * np.eye(pp)
    m = np.zeros(J);  v = np.ones(J) * 0.1
    a_e = 2.0;  b_e = 1.0;  a_u = 2.0;  b_u = 1.0
    for _ in range(n_iter):
        E_tau_e = a_e / b_e;  E_tau_u = a_u / b_u
        y_tilde    = y - m[groups]
        Sigma_beta = np.linalg.inv(E_tau_e * X.T @ X + LAM4 * np.eye(pp))
        mu_beta    = E_tau_e * Sigma_beta @ X.T @ y_tilde
        r = y - X @ mu_beta
        for j in range(J):
            idx_j  = groups == j
            v[j]   = 1.0 / (E_tau_e * n_groups[j] + E_tau_u)
            m[j]   = v[j] * E_tau_e * np.sum(r[idx_j])
        resid = y - X @ mu_beta - m[groups]
        XSX   = np.einsum('ij,jk,ik->i', X, Sigma_beta, X)
        a_e   = ALPHA_E4 + N / 2
        b_e   = GAMMA_E4 + 0.5*(np.sum(resid**2) + np.sum(XSX) + np.sum(v[groups]))
        a_u   = ALPHA_U4 + J / 2
        b_u   = GAMMA_U4 + 0.5*np.sum(m**2 + v)
    return mu_beta, Sigma_beta, m, v, a_e, b_e, a_u, b_u

mu4, Sig4_vb, m4, v4, ae4, be4, au4, bu4 = _warm_cavi_cs4(
    X4, y4, groups4, J4, n_groups4)
E_tau_e4 = ae4 / be4
E_tau_u4 = au4 / bu4

# Load Gibbs reference
df4g = pl.read_parquet(DATA / 'hierarchical_linear_gibbs.parquet')
g4_sd  = {col: df4g[col].to_numpy().std(ddof=1) for col in df4g.columns}
g4_mu  = {col: df4g[col].to_numpy().mean() for col in df4g.columns}

print(f'CS4 CAVI: mu_beta={mu4.round(4)},  E[tau_e]={E_tau_e4:.4f},  E[tau_u]={E_tau_u4:.4f}')
print(f'CS4 CAVI: SD_beta0={np.sqrt(Sig4_vb[0,0]):.4f},  SD_beta1={np.sqrt(Sig4_vb[1,1]):.4f}')
print(f'CS4 Gibbs: SD_beta0={g4_sd["beta0"]:.4f},  SD_beta1={g4_sd["beta1"]:.4f}')
print(f'CS4 SD ratios (CAVI/Gibbs): beta0={np.sqrt(Sig4_vb[0,0])/g4_sd["beta0"]:.3f},  beta1={np.sqrt(Sig4_vb[1,1])/g4_sd["beta1"]:.3f}')

CS4 CAVI: mu_beta=[0.4604 1.9877],  E[tau_e]=3.6959,  E[tau_u]=3.2818
CS4 CAVI: SD_beta0=0.0425,  SD_beta1=0.0368
CS4 Gibbs: SD_beta0=0.3270,  SD_beta1=0.0373
CS4 SD ratios (CAVI/Gibbs): beta0=0.130,  beta1=0.987


In [6]:
# Stage 2b — Laplace correction for CS4
# Parameters: theta = [beta0, beta1, u0..u4, log_tau_e, log_tau_u] — 9D

def log_post_cs4(theta):
    """Log-posterior for CS4 in unconstrained space.
    theta = [beta0, beta1, u0, u1, u2, u3, u4, log_tau_e, log_tau_u]
    Priors: beta ~ N(0, LAM^-1 I), tau_e ~ Gamma(1, 0.1), tau_u ~ Gamma(1, 0.1)
    """
    beta  = theta[:2]
    u     = theta[2:2+J4]
    ell_e = theta[2+J4]
    ell_u = theta[2+J4+1]
    tau_e = np.exp(ell_e)
    tau_u = np.exp(ell_u)
    r = y4 - X4 @ beta - u[groups4]

    # Log-likelihood
    ll = N4/2 * ell_e - tau_e/2 * np.dot(r, r)
    # Beta prior: N(0, LAM^{-1} I)
    lp_beta = -LAM4/2 * np.dot(beta, beta)
    # Random effects: N(0, tau_u^{-1})
    lp_u = J4/2 * ell_u - tau_u/2 * np.dot(u, u)
    # tau_e prior: Gamma(1, 0.1) in log space (Jacobian included)
    # log p(ell_e) = alpha_e * ell_e - gamma_e * tau_e
    lp_tau_e = ALPHA_E4 * ell_e - GAMMA_E4 * tau_e
    # tau_u prior: Gamma(1, 0.1) in log space
    lp_tau_u = ALPHA_U4 * ell_u - GAMMA_U4 * tau_u

    return ll + lp_beta + lp_u + lp_tau_e + lp_tau_u

# CAVI expansion point
theta4_cavi = np.concatenate([mu4, m4, [np.log(E_tau_e4), np.log(E_tau_u4)]])

t0 = time.perf_counter()
Sigma4_corr, H4 = laplace_sigma(log_post_cs4, theta4_cavi, h=1e-4)
t1 = time.perf_counter()

sd4_vb   = np.sqrt(np.diag(Sig4_vb))         # CAVI SD for beta
sd4_corr = np.sqrt(np.diag(Sigma4_corr)[:2]) # Laplace SD for beta
sd4_gibbs = np.array([g4_sd['beta0'], g4_sd['beta1']])

print(f'CS4 Laplace correction ({t1-t0:.2f} s):')
print(f'  SD_beta0: CAVI={sd4_vb[0]:.4f},  Laplace={sd4_corr[0]:.4f},  Gibbs={sd4_gibbs[0]:.4f}')
print(f'  SD_beta1: CAVI={sd4_vb[1]:.4f},  Laplace={sd4_corr[1]:.4f},  Gibbs={sd4_gibbs[1]:.4f}')
print(f'  SD ratio (Laplace/Gibbs): beta0={sd4_corr[0]/sd4_gibbs[0]:.3f},  beta1={sd4_corr[1]/sd4_gibbs[1]:.3f}')
print(f'  SD ratio (CAVI/Gibbs):    beta0={sd4_vb[0]/sd4_gibbs[0]:.3f},  beta1={sd4_vb[1]/sd4_gibbs[1]:.3f}')

CS4 Laplace correction (0.00 s):
  SD_beta0: CAVI=0.0425,  Laplace=0.2497,  Gibbs=0.3270
  SD_beta1: CAVI=0.0368,  Laplace=0.0372,  Gibbs=0.0373
  SD ratio (Laplace/Gibbs): beta0=0.764,  beta1=0.998
  SD ratio (CAVI/Gibbs):    beta0=0.130,  beta1=0.987


In [7]:
# Stage 2c — CS4 CI comparison figure
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
param_names4 = [r'$\beta_0$', r'$\beta_1$']
gibbs4_cols  = ['beta0', 'beta1']

for i, (pname, gcol) in enumerate(zip(param_names4, gibbs4_cols)):
    ax = axes[i]
    g_samp = df4g[gcol].to_numpy()
    x_lo = g_samp.mean() - 4*g4_sd[gcol]
    x_hi = g_samp.mean() + 4*g4_sd[gcol]
    xv = np.linspace(x_lo, x_hi, 400)

    ax.hist(g_samp, bins=60, density=True, color='#E7298A', alpha=0.30,
            label='Gibbs', edgecolor='none')
    ax.plot(xv, stats.norm.pdf(xv, mu4[i], sd4_vb[i]),
            lw=1.8, color='steelblue', ls='--', label='CAVI (collapsed)')
    ax.plot(xv, stats.norm.pdf(xv, mu4[i], sd4_corr[i]),
            lw=2.0, color='darkorange', ls='-', label='Laplace corrected')

    ax.set_xlabel(pname, fontsize=12)
    ax.set_ylabel('Density')
    ax.set_title(f'CS4 — posterior {pname}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('CS4 (hierarchical linear): CAVI vs Laplace vs Gibbs', fontsize=11, y=1.01)
plt.tight_layout()
save_fig(fig, 'correction_cs4_posterior.png')
print('Saved: correction_cs4_posterior.png')

  Saved: correction_cs4_posterior.png
Saved: correction_cs4_posterior.png


---
## Stage 3 — CS5: Hierarchical Bayesian Logistic Regression

Model: $y_{ij} \sim \text{Bernoulli}(\sigma(\mathbf{x}_{ij}^T\boldsymbol{\beta} + u_j))$,  
$u_j \sim \mathcal{N}(0, \tau_u^{-1})$, $\boldsymbol{\beta} \sim \mathcal{N}(0, \lambda^{-1}I)$,  
$\tau_u \sim \text{Gamma}(\alpha_u, \gamma_u)$.  
Parameters: $\lambda=0.1$, $\alpha_u=1$, $\gamma_u=0.1$.

Unconstrained: $(\boldsymbol{\beta}, \mathbf{u}, \ell_u)$ — 8D.

In [8]:
# Stage 3a — Load CS5 data and run CAVI
df5 = pl.read_parquet(DATA / 'hierarchical_logistic_data.parquet')
X5      = np.column_stack([df5['X0'].to_numpy(), df5['X1'].to_numpy()])
y5      = df5['y'].to_numpy()
groups5 = df5['group'].to_numpy()
N5, p5  = X5.shape
J5      = int(groups5.max()) + 1
n_groups5 = np.array([np.sum(groups5 == j) for j in range(J5)])

LAM5    = 0.1
ALPHA_U5, GAMMA_U5 = 1.0, 0.1

def _warm_cavi_cs5(X, y, groups, J, n_groups, n_iter=120):
    N, pp = X.shape
    mu_beta    = np.zeros(pp)
    Sigma_beta = np.eye(pp)
    m  = np.zeros(J);  v = np.ones(J) * 0.5
    a_u = 2.0;  b_u = 1.0
    for _ in range(n_iter):
        E_tau_u = a_u / b_u
        psi_mean = X @ mu_beta + m[groups]
        XSX      = np.einsum('ij,jk,ik->i', X, Sigma_beta, X)
        s2       = XSX + v[groups]
        xi       = np.maximum(np.sqrt(psi_mean**2 + s2), 1e-10)
        lam_ij   = np.tanh(xi / 2.) / (4. * xi)
        Sigma_beta = np.linalg.inv(LAM5 * np.eye(pp) + 2. * (X.T * lam_ij) @ X)
        rhs        = np.sum((y - 0.5 - 2. * lam_ij * m[groups])[:, None] * X, axis=0)
        mu_beta    = Sigma_beta @ rhs
        psi_beta = X @ mu_beta
        for j in range(J):
            idx_j = groups == j
            v[j]  = 1. / (2. * np.sum(lam_ij[idx_j]) + E_tau_u)
            m[j]  = v[j] * np.sum(y[idx_j] - 0.5 - 2. * lam_ij[idx_j] * psi_beta[idx_j])
        a_u = ALPHA_U5 + J / 2.
        b_u = GAMMA_U5 + 0.5 * np.sum(m**2 + v)
    return mu_beta, Sigma_beta, m, v, a_u, b_u

mu5, Sig5_vb, m5, v5, au5, bu5 = _warm_cavi_cs5(
    X5, y5, groups5, J5, n_groups5)
E_tau_u5 = au5 / bu5

# Load Gibbs reference
df5g = pl.read_parquet(DATA / 'hierarchical_logistic_gibbs.parquet')
g5_sd = {col: df5g[col].to_numpy().std(ddof=1) for col in df5g.columns}
g5_mu = {col: df5g[col].to_numpy().mean() for col in df5g.columns}

print(f'CS5 CAVI: mu_beta={mu5.round(4)},  E[tau_u]={E_tau_u5:.4f}')
print(f'CS5 CAVI: SD_beta0={np.sqrt(Sig5_vb[0,0]):.4f},  SD_beta1={np.sqrt(Sig5_vb[1,1]):.4f}')
print(f'CS5 Gibbs: SD_beta0={g5_sd["beta0"]:.4f},  SD_beta1={g5_sd["beta1"]:.4f}')
print(f'CS5 SD ratios (CAVI/Gibbs): beta0={np.sqrt(Sig5_vb[0,0])/g5_sd["beta0"]:.3f},  beta1={np.sqrt(Sig5_vb[1,1])/g5_sd["beta1"]:.3f}')

CS5 CAVI: mu_beta=[-1.4458  2.202 ],  E[tau_u]=10.0545
CS5 CAVI: SD_beta0=0.1991,  SD_beta1=0.1881
CS5 Gibbs: SD_beta0=0.4298,  SD_beta1=0.4098
CS5 SD ratios (CAVI/Gibbs): beta0=0.463,  beta1=0.459


In [9]:
# Stage 3b — Laplace correction for CS5
# Parameters: theta = [beta0, beta1, u0..u4, log_tau_u] — 8D

def log_post_cs5(theta):
    """Log-posterior for CS5 in unconstrained space.
    theta = [beta0, beta1, u0, u1, u2, u3, u4, log_tau_u]
    Priors: beta ~ N(0, LAM^-1 I), tau_u ~ Gamma(1, 0.1)
    Likelihood: Bernoulli logistic
    """
    beta  = theta[:2]
    u     = theta[2:2+J5]
    ell_u = theta[2+J5]
    tau_u = np.exp(ell_u)
    psi   = X5 @ beta + u[groups5]

    # Log-likelihood: sum y_ij * psi_ij - log(1 + exp(psi_ij))
    # Use numerically stable: log1p(exp(psi)) for large psi
    log1pexp = np.where(psi > 20, psi, np.log1p(np.exp(psi)))
    ll = np.sum(y5 * psi - log1pexp)

    # Beta prior
    lp_beta = -LAM5/2 * np.dot(beta, beta)
    # Random effects
    lp_u = J5/2 * ell_u - tau_u/2 * np.dot(u, u)
    # tau_u prior in log space (Jacobian included)
    lp_tau_u = ALPHA_U5 * ell_u - GAMMA_U5 * tau_u

    return ll + lp_beta + lp_u + lp_tau_u

theta5_cavi = np.concatenate([mu5, m5, [np.log(E_tau_u5)]])

t0 = time.perf_counter()
Sigma5_corr, H5 = laplace_sigma(log_post_cs5, theta5_cavi, h=1e-4)
t1 = time.perf_counter()

sd5_vb   = np.sqrt(np.diag(Sig5_vb))
sd5_corr = np.sqrt(np.diag(Sigma5_corr)[:2])
sd5_gibbs = np.array([g5_sd['beta0'], g5_sd['beta1']])

print(f'CS5 Laplace correction ({t1-t0:.2f} s):')
print(f'  SD_beta0: CAVI={sd5_vb[0]:.4f},  Laplace={sd5_corr[0]:.4f},  Gibbs={sd5_gibbs[0]:.4f}')
print(f'  SD_beta1: CAVI={sd5_vb[1]:.4f},  Laplace={sd5_corr[1]:.4f},  Gibbs={sd5_gibbs[1]:.4f}')
print(f'  SD ratio (Laplace/Gibbs): beta0={sd5_corr[0]/sd5_gibbs[0]:.3f},  beta1={sd5_corr[1]/sd5_gibbs[1]:.3f}')
print(f'  SD ratio (CAVI/Gibbs):    beta0={sd5_vb[0]/sd5_gibbs[0]:.3f},  beta1={sd5_vb[1]/sd5_gibbs[1]:.3f}')

CS5 Laplace correction (0.00 s):
  SD_beta0: CAVI=0.1991,  Laplace=0.3525,  Gibbs=0.4298
  SD_beta1: CAVI=0.1881,  Laplace=0.3588,  Gibbs=0.4098
  SD ratio (Laplace/Gibbs): beta0=0.820,  beta1=0.876
  SD ratio (CAVI/Gibbs):    beta0=0.463,  beta1=0.459


In [10]:
# Stage 3c — CS5 CI comparison figure
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
param_names5 = [r'$\beta_0$', r'$\beta_1$']
gibbs5_cols  = ['beta0', 'beta1']

for i, (pname, gcol) in enumerate(zip(param_names5, gibbs5_cols)):
    ax = axes[i]
    g_samp = df5g[gcol].to_numpy()
    x_lo = g_samp.mean() - 4*g5_sd[gcol]
    x_hi = g_samp.mean() + 4*g5_sd[gcol]
    xv = np.linspace(x_lo, x_hi, 400)

    ax.hist(g_samp, bins=60, density=True, color='#E7298A', alpha=0.30,
            label='Gibbs', edgecolor='none')
    ax.plot(xv, stats.norm.pdf(xv, mu5[i], sd5_vb[i]),
            lw=1.8, color='steelblue', ls='--', label='CAVI (collapsed)')
    ax.plot(xv, stats.norm.pdf(xv, mu5[i], sd5_corr[i]),
            lw=2.0, color='darkorange', ls='-', label='Laplace corrected')

    ax.set_xlabel(pname, fontsize=12)
    ax.set_ylabel('Density')
    ax.set_title(f'CS5 — posterior {pname}', fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('CS5 (hierarchical logistic): CAVI vs Laplace vs Gibbs', fontsize=11, y=1.01)
plt.tight_layout()
save_fig(fig, 'correction_cs5_posterior.png')
print('Saved: correction_cs5_posterior.png')

  Saved: correction_cs5_posterior.png
Saved: correction_cs5_posterior.png


---
## Stage 4 — Cross-Case Summary

In [11]:
# Stage 4a — SD ratio comparison figure (3 cases x 2 beta params)
cases   = ['CS1\n(linear)', 'CS4\n(hier. linear)', 'CS5\n(hier. logistic)']
vb_ratios_b0   = [sd1_vb[0]/g1_sd['beta0'], sd4_vb[0]/g4_sd['beta0'], sd5_vb[0]/g5_sd['beta0']]
corr_ratios_b0 = [sd1_corr[0]/g1_sd['beta0'], sd4_corr[0]/g4_sd['beta0'], sd5_corr[0]/g5_sd['beta0']]
vb_ratios_b1   = [sd1_vb[1]/g1_sd['beta1'], sd4_vb[1]/g4_sd['beta1'], sd5_vb[1]/g5_sd['beta1']]
corr_ratios_b1 = [sd1_corr[1]/g1_sd['beta1'], sd4_corr[1]/g4_sd['beta1'], sd5_corr[1]/g5_sd['beta1']]

x_pos = np.arange(len(cases))
w = 0.28

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, title, vb_rat, corr_rat in zip(
        axes,
        [r'$\beta_0$', r'$\beta_1$'],
        [vb_ratios_b0, vb_ratios_b1],
        [corr_ratios_b0, corr_ratios_b1]):

    ax.bar(x_pos - w/2, vb_rat,   w, label='CAVI',             color='steelblue', alpha=0.85)
    ax.bar(x_pos + w/2, corr_rat, w, label='Laplace corrected', color='darkorange', alpha=0.85)
    ax.axhline(1.0, color='black', lw=1.2, ls='--', label='Ideal (= Gibbs)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(cases, fontsize=9)
    ax.set_ylabel(r'$\sigma_{\mathrm{VB}} / \sigma_{\mathrm{Gibbs}}$')
    ax.set_title(f'SD ratio: {title}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylim(0, max(max(corr_rat), 1.3))

plt.suptitle('Posterior SD ratios: CAVI vs Laplace correction vs Gibbs', fontsize=11, y=1.01)
plt.tight_layout()
save_fig(fig, 'correction_sd_ratios_summary.png')
print('Saved: correction_sd_ratios_summary.png')

  Saved: correction_sd_ratios_summary.png
Saved: correction_sd_ratios_summary.png


In [12]:
# Stage 4b — Save summary table (LaTeX)
rows = [
    # (case, param, cavi_sd, corr_sd, gibbs_sd, cavi_ratio, corr_ratio)
    ('CS1', r'$\beta_0$', sd1_vb[0], sd1_corr[0], g1_sd['beta0']),
    ('CS1', r'$\beta_1$', sd1_vb[1], sd1_corr[1], g1_sd['beta1']),
    ('CS4', r'$\beta_0$', sd4_vb[0], sd4_corr[0], g4_sd['beta0']),
    ('CS4', r'$\beta_1$', sd4_vb[1], sd4_corr[1], g4_sd['beta1']),
    ('CS5', r'$\beta_0$', sd5_vb[0], sd5_corr[0], g5_sd['beta0']),
    ('CS5', r'$\beta_1$', sd5_vb[1], sd5_corr[1], g5_sd['beta1']),
]

print('Summary table:')
print(f'{"Case":4s} {"Param":10s} {"CAVI SD":10s} {"Laplace SD":12s} {"Gibbs SD":10s} {"CAVI ratio":12s} {"Laplace ratio":13s}')
for case, param, cavi_sd, corr_sd, gibbs_sd in rows:
    print(f'{case:4s} {param:10s} {cavi_sd:10.4f} {corr_sd:12.4f} {gibbs_sd:10.4f} '
          f'{cavi_sd/gibbs_sd:12.3f} {corr_sd/gibbs_sd:13.3f}')

# Save LaTeX table
tex_path = TABLES / 'correction_summary.tex'
with open(tex_path, 'w') as f:
    f.write('\\begin{tblr}{colspec={llrrrrr}, hline{1,2,Z}={solid}, hline{4,6}={dashed}}'  + '\n')
    f.write('  Case & Param. & CAVI SD & Laplace SD & Gibbs SD & CAVI/Gibbs & Laplace/Gibbs \\\\\n')
    for case, param, cavi_sd, corr_sd, gibbs_sd in rows:
        f.write(f'  {case} & {param} & {cavi_sd:.4f} & {corr_sd:.4f} & {gibbs_sd:.4f} '
                f'& {cavi_sd/gibbs_sd:.3f} & {corr_sd/gibbs_sd:.3f} \\\\\n')
    f.write('\\end{tblr}\n')
print(f'Saved: {tex_path.name}')
print('\nAll outputs complete.')

Summary table:
Case Param      CAVI SD    Laplace SD   Gibbs SD   CAVI ratio   Laplace ratio
CS1  $\beta_0$      0.1101       0.1101     0.1124        0.979         0.979
CS1  $\beta_1$      0.0379       0.0379     0.0387        0.981         0.981
CS4  $\beta_0$      0.0425       0.2497     0.3270        0.130         0.764
CS4  $\beta_1$      0.0368       0.0372     0.0373        0.987         0.998
CS5  $\beta_0$      0.1991       0.3525     0.4298        0.463         0.820
CS5  $\beta_1$      0.1881       0.3588     0.4098        0.459         0.876
Saved: correction_summary.tex

All outputs complete.
